In [ ]:
import os
import threading
from pathlib import Path
import json
import numpy as np
import pandas as pd
import torch
import yaml
import xarray as xr
import netCDF4
from tqdm.auto import tqdm
from huggingface_hub import hf_hub_download
import shutil
from PrithviWxC.dataloaders.merra2 import (
    Merra2Dataset,
    input_scalers,
    output_scalers,
    preproc,
    static_input_scalers,
)
from PrithviWxC.model import PrithviWxC
from PrithviWxC.download import download_merra_files, extract_prithvi_wxc_input_data, get_prithvi_wxc_climatology, get_prithvi_wxc_input, get_required_input_files
from datetime import datetime

In [2]:
# Paths
ROOT = Path("..").resolve()
DATA_DIR = ROOT / "data"
surf_dir = DATA_DIR / "merra-2"
vert_dir = DATA_DIR / "merra-2"
surf_clim_dir = DATA_DIR / "climatology"
vert_clim_dir = DATA_DIR / "climatology"

# Variables (same pretrained setup)
surface_vars = [
    "EFLUX", "GWETROOT", "HFLUX", "LAI", "LWGAB", "LWGEM", "LWTUP", "PS", "QV2M", "SLP",
    "SWGNT", "SWTNT", "T2M", "TQI", "TQL", "TQV", "TS", "U10M", "V10M", "Z0M",
]
static_surface_vars = ["FRACI", "FRLAND", "FROCEAN", "PHIS"]
vertical_vars = ["CLOUD", "H", "OMEGA", "PL", "QI", "QL", "QV", "T", "U", "V"]
levels = [34.0, 39.0, 41.0, 43.0, 44.0, 45.0, 48.0, 51.0, 53.0, 56.0, 63.0, 68.0, 71.0, 72.0]
padding = {"level": [0, 0], "lat": [0, -1], "lon": [0, 0]}

# Task setup
lead_time = 0
input_delta = -6
time_range = ("1980-01-01T00:00:00", "2021-12-31T23:59:59")

residual = "climate"
masking_mode = "both"
encoder_shifting = True
decoder_shifting = True
positional_encoding = "fourier"
masking_ratio = 0.0

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

Using device: cuda


In [3]:
surf_in_scal_path = Path("../data/climatology/musigma_surface.nc")
hf_hub_download(
    repo_id="ibm-nasa-geospatial/Prithvi-WxC-1.0-2300M",
    filename=f"climatology/{surf_in_scal_path.name}",
    local_dir="../data",
)

vert_in_scal_path = Path("../data/climatology/musigma_vertical.nc")
hf_hub_download(
    repo_id="ibm-nasa-geospatial/Prithvi-WxC-1.0-2300M",
    filename=f"climatology/{vert_in_scal_path.name}",
    local_dir="../data",
)

surf_out_scal_path = Path("../data/climatology/anomaly_variance_surface.nc")
hf_hub_download(
    repo_id="ibm-nasa-geospatial/Prithvi-WxC-1.0-2300M",
    filename=f"climatology/{surf_out_scal_path.name}",
    local_dir="../data",
)

vert_out_scal_path = Path("../data/climatology/anomaly_variance_vertical.nc")
hf_hub_download(
    repo_id="ibm-nasa-geospatial/Prithvi-WxC-1.0-2300M",
    filename=f"climatology/{vert_out_scal_path.name}",
    local_dir="../data",
)

in_mu, in_sig = input_scalers(
    surface_vars,
    vertical_vars,
    levels,
    surf_in_scal_path,
    vert_in_scal_path,
)

output_sig = output_scalers(
    surface_vars,
    vertical_vars,
    levels,
    surf_out_scal_path,
    vert_out_scal_path,
)

static_mu, static_sig = static_input_scalers(
    surf_in_scal_path,
    static_surface_vars,
)

climatology/musigma_surface.nc:   0%|          | 0.00/24.7k [00:00<?, ?B/s]

climatology/musigma_vertical.nc:   0%|          | 0.00/25.0k [00:00<?, ?B/s]

climatology/anomaly_variance_surface.nc:   0%|          | 0.00/11.5k [00:00<?, ?B/s]

climatology/anomaly_variance_vertical.nc:   0%|          | 0.00/18.6k [00:00<?, ?B/s]

In [4]:
# Delete scaling files because already loaded into memory above
for name in ["musigma_surface.nc", "musigma_vertical.nc",
             "anomaly_variance_surface.nc", "anomaly_variance_vertical.nc"]:
    p = Path("../data/climatology") / name
    if p.exists():
        p.unlink()

In [5]:
# Model
hf_hub_download(
    repo_id="ibm-nasa-geospatial/Prithvi-WxC-1.0-2300M",
    filename="config.yaml",
    local_dir="../data",
)

with open("../data/config.yaml", "r") as f:
    config = yaml.safe_load(f)

model = PrithviWxC(
    in_channels=config["params"]["in_channels"],
    input_size_time=config["params"]["input_size_time"],
    in_channels_static=config["params"]["in_channels_static"],
    input_scalers_mu=in_mu,
    input_scalers_sigma=in_sig,
    input_scalers_epsilon=config["params"]["input_scalers_epsilon"],
    static_input_scalers_mu=static_mu,
    static_input_scalers_sigma=static_sig,
    static_input_scalers_epsilon=config["params"]["static_input_scalers_epsilon"],
    output_scalers=output_sig**0.5,
    n_lats_px=config["params"]["n_lats_px"],
    n_lons_px=config["params"]["n_lons_px"],
    patch_size_px=config["params"]["patch_size_px"],
    mask_unit_size_px=config["params"]["mask_unit_size_px"],
    mask_ratio_inputs=masking_ratio,
    mask_ratio_targets=0.0,
    embed_dim=config["params"]["embed_dim"],
    n_blocks_encoder=config["params"]["n_blocks_encoder"],
    n_blocks_decoder=config["params"]["n_blocks_decoder"],
    mlp_multiplier=config["params"]["mlp_multiplier"],
    n_heads=config["params"]["n_heads"],
    dropout=config["params"]["dropout"],
    drop_path=config["params"]["drop_path"],
    parameter_dropout=config["params"]["parameter_dropout"],
    residual=residual,
    masking_mode=masking_mode,
    encoder_shifting=encoder_shifting,
    decoder_shifting=decoder_shifting,
    positional_encoding=positional_encoding,
    checkpoint_encoder=[],
    checkpoint_decoder=[],
)

weights_path = Path("../data/weights/prithvi.wxc.2300m.v1.pt")
hf_hub_download(
    repo_id="ibm-nasa-geospatial/Prithvi-WxC-1.0-2300M",
    filename=weights_path.name,
    local_dir="../data/weights",
)
state_dict = torch.load(weights_path, weights_only=False)
if "model_state" in state_dict:
    state_dict = state_dict["model_state"]
model.load_state_dict(state_dict, strict=True)

# delete model weights file because it already loaded to save space
# del state_dict
# weights_path.unlink()
# print(f"Deleted {weights_path} to save disk space")

if (hasattr(model, "device") and model.device != device) or not hasattr(model, "device"):
    model = model.to(device)

model.mask_ratio_inputs = 0.0
print("Model ready.")

Model ready.


In [ ]:
# Daily timestamps: 1982 year, months 5-9 (May-Sep), 12:00 UTC
days = pd.date_range("2012-05-01", "2012-09-30", freq="D")
days = days[days.month.isin([5,6,7,8])]
timestamps = days + pd.Timedelta(hours=12)

In [ ]:
output_dir = Path("latent_output")
output_dir.mkdir(exist_ok=True)

for ts in tqdm(timestamps, desc="Extracting latents",leave=False):
    get_prithvi_wxc_input(
        np.datetime64(ts),
        input_time_step=6,
        lead_time=lead_time,
        input_data_dir=str(DATA_DIR / "merra-2"),
        download_dir=str(DATA_DIR / "raw")
    )

    dataset = Merra2Dataset(
        time_range=(str(ts), str(ts)),
        lead_times=[lead_time],
        input_times=[input_delta],
        data_path_surface=surf_dir,
        data_path_vertical=vert_dir,
        climatology_path_surface=surf_clim_dir,
        climatology_path_vertical=vert_clim_dir,
        surface_vars=surface_vars,
        static_surface_vars=static_surface_vars,
        vertical_vars=vertical_vars,
        levels=levels,
        positional_encoding=positional_encoding,
    )

    nested = Path("../data/climatology/climatology")
    root = Path("../data/climatology")
    if nested.exists():
        for f in nested.iterdir():
            dest = root / f.name
            if dest.exists():
                dest.unlink()
            shutil.move(str(f), root)
        if not any(nested.iterdir()):
            nested.rmdir()
    data = dataset.get_data(timestamp=ts, input_time=input_delta, lead_time=lead_time)

    batch = preproc([data], padding)

    for k, v in batch.items():
        if isinstance(v, torch.Tensor):
            batch[k] = v.to(device)

    latent = {}

    def save_encoder_output(module, inp, out):
        latent["x_encoded"] = out.detach().cpu()

    def save_decoder_output(module, inp, out):
        latent["x_decoded"] = out.detach().cpu()

    h_enc = model.encoder.register_forward_hook(save_encoder_output)
    h_dec = model.decoder.register_forward_hook(save_decoder_output)

    with torch.no_grad():
        model.eval()
        _ = model(batch)

    h_enc.remove()
    h_dec.remove()

    lats = dataset.lats
    lons = dataset.lons

    patch_h, patch_w = model.patch_size_px
    g_lat, g_lon = model.global_shape_mu
    l_lat, l_lon = model.local_shape_mu

    n_patch_lat = g_lat * l_lat
    n_patch_lon = g_lon * l_lon

    lat_patch = lats[:n_patch_lat * patch_h].reshape(n_patch_lat, patch_h).mean(axis=1)
    lon_patch = lons[:n_patch_lon * patch_w].reshape(n_patch_lon, patch_w).mean(axis=1)

    lat2d, lon2d = np.meshgrid(lat_patch, lon_patch, indexing="ij")
    patch_mask = (
        (lat2d >= -10) & (lat2d <= 35) &
        (lon2d >= 40) & (lon2d <= 100)
    )

    token_mask = (
        torch.from_numpy(patch_mask)
        .view(g_lat, l_lat, g_lon, l_lon)
        .permute(0, 2, 1, 3)
        .reshape(g_lat * g_lon, l_lat * l_lon)
    )

    tokens_e = latent["x_encoded"][0]
    tokens_d = latent["x_decoded"][0]
    region_tokens_e = tokens_e[token_mask]
    region_tokens_d = tokens_d[token_mask]
    daily_vec_e = region_tokens_e.mean(dim=0).numpy()
    daily_vec_d = region_tokens_d.mean(dim=0).numpy()

    for d in [DATA_DIR / "raw", DATA_DIR / "merra-2"]:
        if d.exists():
            shutil.rmtree(d)

    year_dir = output_dir / f"{ts.year}"
    year_dir.mkdir(parents=True, exist_ok=True)
    np.save(year_dir / f"encoder_{ts}.npy", daily_vec_e)
    np.save(year_dir / f"decoder_{ts}.npy", daily_vec_d)